In [3]:
import polars as pl

# 1. Lendo os arquivos (ajuste o caminho se necessário)
print("🔍 Vasculhando os arquivos em busca dos artilheiros...")
df = pl.read_parquet("shots_preparados.parquet")

# 2. Contagem de chutes por jogador
top_jogadores = (
    df.group_by("player_name")
    .agg([
        pl.len().alias("total_chutes"),
        pl.col("is_goal").sum().alias("gols_reais"),
        pl.col("shot_statsbomb_xg").sum().alias("xg_acumulado")
    ])
    .filter(pl.col("total_chutes") > 50) # Filtro de relevância: pelo menos 50 chutes
    .with_columns([
        (pl.col("gols_reais") / pl.col("xg_acumulado")).alias("fator_eficiencia")
    ])
    .sort("total_chutes", descending=True)
)

print(top_jogadores.head(10))

🔍 Vasculhando os arquivos em busca dos artilheiros...
shape: (10, 5)
┌─────────────────────────────────┬──────────────┬────────────┬──────────────┬──────────────────┐
│ player_name                     ┆ total_chutes ┆ gols_reais ┆ xg_acumulado ┆ fator_eficiencia │
│ ---                             ┆ ---          ┆ ---        ┆ ---          ┆ ---              │
│ str                             ┆ u32          ┆ u32        ┆ f64          ┆ f64              │
╞═════════════════════════════════╪══════════════╪════════════╪══════════════╪══════════════════╡
│ Lionel Andrés Messi Cuccittini  ┆ 2670         ┆ 510        ┆ 363.372823   ┆ 1.403517         │
│ Luis Alberto Suárez Díaz        ┆ 638          ┆ 140        ┆ 111.744621   ┆ 1.252857         │
│ Neymar da Silva Santos Junior   ┆ 466          ┆ 89         ┆ 77.399037    ┆ 1.149885         │
│ Cristiano Ronaldo dos Santos A… ┆ 413          ┆ 59         ┆ 59.089969    ┆ 0.998477         │
│ Andrés Iniesta Luján            ┆ 378          

In [1]:
import polars as pl
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import brier_score_loss, roc_auc_score

# ====================== 1. CONFIGURAÇÕES E CONSTANTES ======================
NASCIMENTOS = {
    "Lionel Andrés Messi Cuccittini": "1987-06-24",
    "Luis Alberto Suárez Díaz": "1987-01-24"
}
GOAL_CENTER = (120, 40)

# ====================== 2. CARGA E LIMPEZA (CAMADA SILVER) ======================
print("📖 Lendo Camada Silver (12M eventos)...")
df_pl = pl.read_parquet("statsbomb_completo.parquet")

# Normaliza nomes de colunas (substitui . por _)
df_pl = df_pl.rename({col: col.replace(".", "_") for col in df_pl.columns})

# Filtramos apenas o essencial para o modelo
cols_selecionadas = [
    "match_id", "player_name", "type_name", "location", 
    "under_pressure", "shot_statsbomb_xg", "shot_outcome_name", 
    "shot_body_part_name", "shot_technique_name"
]
df_pl = df_pl.select([c for c in cols_selecionadas if c in df_pl.columns])

# ====================== 3. ENGENHARIA DE CONTEXTO (5 EVENTOS) ======================
print("🧠 Analisando contexto de pressão (Janela de 5 eventos)...")

# Transformamos pressão em binário e calculamos a média móvel dos últimos 5 eventos por jogo
df_pl = df_pl.with_columns([
    pl.col("under_pressure").fill_null(False).cast(pl.Int8).alias("pressao_binaria")
]).with_columns([
    pl.col("pressao_binaria").rolling_mean(window_size=5).over("match_id").alias("pressao_contexto_5_eventos")
])

# ====================== 4. GEOMETRIA VETORIZADA (A CIÊNCIA) ======================
def aplicar_geometria_padrão(df_pl):
    print("📏 Extraindo coordenadas e calculando geometria (Modo Turbo)...")
    
    # 1. Extração Nativa (Sem lambdas, sem lentidão)
    # Pegamos o primeiro item da lista [0] para X e o segundo [1] para Y
    df_pl = df_pl.with_columns([
        pl.col("location").list.get(0).alias("x"),
        pl.col("location").list.get(1).alias("y")
    ])
    
    # 2. Tratando Nulos antes de converter para NumPy
    # Se não tiver localização, chutamos o centro do gol (120, 40) para não quebrar a conta
    df_pl = df_pl.with_columns([
        pl.col("x").fill_null(120.0),
        pl.col("y").fill_null(40.0)
    ])
    
    # 3. Convertendo para NumPy para os cálculos matemáticos complexos
    x = df_pl["x"].to_numpy()
    y = df_pl["y"].to_numpy()
    
    # Distâncias e Ângulos
    dist = np.sqrt((120 - x)**2 + (40 - y)**2)
    a = np.sqrt((120 - x)**2 + (36 - y)**2)
    b = np.sqrt((120 - x)**2 + (44 - y)**2)
    
    # Lei dos Cossenos: c^2 = a^2 + b^2 - 2ab*cos(theta)
    cos_theta = np.clip((a**2 + b**2 - 8**2) / (2 * a * b), -1.0, 1.0)
    angulo = np.degrees(np.arccos(cos_theta))
    
    # 4. Devolvendo os resultados para o Polars
    return df_pl.with_columns([
        pl.Series("distancia", dist),
        pl.Series("angulo_visao", angulo),
        (pl.Series("distancia", dist) * (1 + pl.col("pressao_contexto_5_eventos"))).alias("dificuldade_relativa")
    ])

df_gold = aplicar_geometria_padrão(df_pl)

# ====================== 5. TREINAMENTO DO MODELO XGBOOST ======================
# Filtramos apenas os chutes e passamos para Pandas (ML)
print("🚀 Preparando Treino de Elite...")
df_shots = df_gold.filter(pl.col("type_name") == "Shot").to_pandas()
df_shots['is_goal'] = (df_shots['shot_outcome_name'] == 'Goal').astype(int)

# Dummies e Seleção de Features
df_model = pd.get_dummies(df_shots, columns=['shot_body_part_name'], drop_first=True)
features = ['distancia', 'angulo_visao', 'pressao_contexto_5_eventos', 'dificuldade_relativa'] 
features += [c for c in df_model.columns if 'shot_body_part_name_' in c]

X = df_model[features].fillna(0)
y = df_model['is_goal']
groups = df_model['match_id']

# GroupKFold para garantir que não testamos em jogos que treinamos
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    pos_weight = y_tr.value_counts()[0] / y_tr.value_counts()[1]
    
    # --- MUDANÇA AQUI: early_stopping_rounds agora é no construtor ---
    model = xgb.XGBClassifier(
        n_estimators=1000, 
        max_depth=7, 
        learning_rate=0.01,
        scale_pos_weight=pos_weight, 
        reg_lambda=10, 
        tree_method='hist',
        early_stopping_rounds=50,  # Movido para cá
        eval_metric='aucpr'         # Recomendado definir a métrica aqui também
    )
    
    # O .fit() agora fica mais limpo
    model.fit(
        X_tr, y_tr, 
        eval_set=[(X_te, y_te)], 
        verbose=False
    )
    
    # Métrica de Aura (Predição IA vs StatsBomb)
    df_model.loc[test_idx, 'prob_ia'] = model.predict_proba(X_te)[:, 1]

# Cálculo Final da Aura
df_model['aura'] = df_model['prob_ia'] - df_model['shot_statsbomb_xg']

print("\n" + "="*40)
print(f"✅ TREINO CONCLUÍDO!")
print(f"📊 Média de Aura (Geral): {df_model['aura'].mean():.4f}")
print(f"📊 ROC-AUC: {roc_auc_score(y, df_model['prob_ia']):.4f}")
print("="*40)

# ====================== 6. RANKING DE AURA (MSN vs O MUNDO) ======================
ranking_aura = df_model.groupby('player_name')['aura'].mean().sort_values(ascending=False).head(10)
print("\n🔥 TOP 10 JOGADORES COM MAIOR AURA (Eficiência além do esperado):")
print(ranking_aura)

📖 Lendo Camada Silver (12M eventos)...
🧠 Analisando contexto de pressão (Janela de 5 eventos)...
📏 Extraindo coordenadas e calculando geometria (Modo Turbo)...
🚀 Preparando Treino de Elite...

✅ TREINO CONCLUÍDO!
📊 Média de Aura (Geral): 0.3011
📊 ROC-AUC: 0.7859

🔥 TOP 10 JOGADORES COM MAIOR AURA (Eficiência além do esperado):
player_name
Carlos Alberto Gomes de Jesus    0.728430
Marcus Andreas Danielsson        0.705118
Fábio Pereira da Silva           0.683788
Caitlin Dijkstra                 0.679031
Pape Maly Diamanka               0.675336
Megan Wynne                      0.665132
Cédric Hountondji                0.664339
Bruno N"Gotty                    0.659062
Bikash Jairu                     0.655476
Luc Rollet De Fougerolles        0.650269
Name: aura, dtype: float64


In [4]:
# Verifique se você não chamou o modelo de 'best_model' ou algo assim
# Se o seu loop terminou, a variável 'model' deve conter o último fold treinado.

try:
    model.save_model("model.json")
    print("✅ Sucesso! O arquivo 'model.json' foi criado.")
    
    # Salva também o dataframe com os cálculos de Aura
    df_model.to_parquet("df_model.parquet")
    print("💾 DataFrame de Aura salvo para consulta rápida!")
    
except NameError:
    print("❌ Erro: A variável 'model' não existe nesta célula.")
    print("👉 Dica: Volte na célula de treino e veja se o nome é 'model' ou 'model'.")

✅ Sucesso! O arquivo 'model.json' foi criado.
💾 DataFrame de Aura salvo para consulta rápida!


In [5]:
import polars as pl

print("📖 Lendo o arquivo gigante de 12M de eventos...")
df = pl.read_parquet("statsbomb_completo.parquet")

# 1. Limpamos os nomes das colunas (trocando ponto por underline)
df = df.rename({col: col.replace(".", "_") for col in df.columns})

print("🎯 Filtrando apenas os chutes para a API...")
# 2. Filtramos apenas o que é chute (Shot)
# Isso reduz o arquivo de 12 milhões de linhas para cerca de 40 mil
df_shots = df.filter(pl.col("type_name") == "Shot")

# 3. Selecionamos só as colunas que o seu main.py pede
cols_api = [
    "match_id", "player_name", "minute", "location", "distancia", 
    "angulo_visao", "under_pressure", "shot_statsbomb_xg", "shot_outcome_name"
]

# Nota: Se você ainda não calculou distancia/angulo, use o select das que existem:
df_final = df_shots.select([
    "match_id", "player_name", "minute", "location", 
    "under_pressure", "shot_statsbomb_xg", "shot_outcome_name"
])

# 4. Criamos a coluna is_goal para facilitar a vida da API
df_final = df_final.with_columns([
    (pl.col("shot_outcome_name") == "Goal").alias("is_goal"),
    pl.col("under_pressure").fill_null(False).alias("sob_pressao")
])

# 5. SALVAMOS O ARQUIVO QUE A API QUER
df_final.write_parquet("shots_preparados.parquet")

print("✅ Arquivo 'shots_preparados.parquet' criado com sucesso!")
print(f"📉 O tamanho caiu de 1.1GB para cerca de 10MB. Agora a API vai voar!")

📖 Lendo o arquivo gigante de 12M de eventos...
🎯 Filtrando apenas os chutes para a API...
✅ Arquivo 'shots_preparados.parquet' criado com sucesso!
📉 O tamanho caiu de 1.1GB para cerca de 10MB. Agora a API vai voar!
